In [1]:
import sys
sys.path.append('..')
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np
from utils.plotter import Plotter
import pandas as pd

/home/kkarthikeyan/deep-learning/DeepLearningProject/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Accuracy and Confusion Matrix

In [2]:
csv_path = "../results_llava-hf/llava-1.5-7b-hf/compute_accuracy/simple_results_all_levels.csv"
results = pd.read_csv(csv_path)
print(results.shape)

(3352, 11)


In [3]:


# Compute accuracy
accuracy = accuracy_score(results['ground_truth'], results['prediction'])
print(f"Overall Accuracy: {accuracy:.4f}")

# Compute confusion matrix
cm_all = pd.crosstab(
        results["ground_truth"],
        results["prediction"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    ).reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)

# print true positive, true negative, false positve, falsne negative rate along with recall and precision
tp = cm_all.loc["yes", "yes"]
fn = cm_all.loc["yes", "no"]
fp = cm_all.loc["no", "yes"]
tn = cm_all.loc["no", "no"]

tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # recall
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
tnr = tn / (tn + fp) if (tn + fp) > 0 else 0  # specificity

precision = tp / (tp + fp) if (tp + fp) > 0 else 0

print(f"True Positive: {tp}, False Negative: {fn}, False Positive: {fp}, True Negative: {tn}")
print(f"True Positive Rate (Recall): {tpr:.4f}")
print(f"False Positive Rate: {fpr:.4f}")
print(f"False Negative Rate: {fnr:.4f}")
print(f"True Negative Rate (Specificity): {tnr:.4f}")
print(f"Precision: {precision:.4f}")

# Plot using Plotter
plotter = Plotter(experiment_name="compute_accuracy")
plotter.plot_confusion_matrix(cm_all, filename="confusion_matrix.png")

Overall Accuracy: 0.5131
True Positive: 1662, False Negative: 14, False Positive: 1618, True Negative: 58
True Positive Rate (Recall): 0.9916
False Positive Rate: 0.9654
False Negative Rate: 0.0084
True Negative Rate (Specificity): 0.0346
Precision: 0.5067


## Analsying it by level

In [4]:
print(results.keys())

Index(['level_id', 'image_id', 'qa_id', 'question', 'ground_truth',
       'relation_type', 'response', 'prediction', 'confidence',
       'num_image_tokens', 'num_text_tokens'],
      dtype='object')


In [5]:
# Group the results by level_id and then look at the accuracy and the confusion
import matplotlib.pyplot as plt

# Compute accuracy by level
results["correct"] = results["prediction"] == results["ground_truth"]
accuracy_by_level = results.groupby("level_id")["correct"].mean()

# plots (accuracy bars + confusion matrix), no CSV output

# Accuracy bar plot
plotter.plot_accuracy_bars(
    accuracy_by_level,
    filename="accuracy_by_level.png",
    title="Accuracy by level",
)

# Assuming 'level_id' is a column in results
levels = results['level_id'].unique()
for level in sorted(levels):
    level_results = results[results['level_id'] == level]
    accuracy = accuracy_score(level_results['ground_truth'], level_results['prediction'])
    print(f"Accuracy for {level}: {accuracy:.4f}")
    
    cm = pd.crosstab(
        level_results["ground_truth"],
        level_results["prediction"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    ).reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    
    # Plot confusion matrix inline
    plotter.plot_confusion_matrix(cm, filename=f"confusion_matrix_{level}.png")

# Compute accuracy by relation_type
accuracy_by_relation = results.groupby("relation_type")["correct"].mean()

# Accuracy bar plot by relation type
plotter.plot_accuracy_bars(
    accuracy_by_relation,
    filename="accuracy_by_relation.png",
    title="Accuracy by Relation Type",
)

# Print accuracies for each relation type
relations = results['relation_type'].unique()
for relation in sorted(relations):
    relation_results = results[results['relation_type'] == relation]
    accuracy = accuracy_score(relation_results['ground_truth'], relation_results['prediction'])
    print(f"Accuracy for {relation}: {accuracy:.4f}")

Accuracy for level_0: 0.5125
Accuracy for level_1: 0.5125
Accuracy for level_2: 0.5405
Accuracy for level_3: 0.5185
Accuracy for level_4: 0.5061
Accuracy for above: 0.5049
Accuracy for below: 0.5194
Accuracy for left_of: 0.5235
Accuracy for right_of: 0.5047


In [6]:
# print the average confidence, and the variance 

# Compare black and white image

In [11]:
# can you compute accuracy, but this time make two groups based on image_id, wiht black ending with b and white ending with w

# Assuming masked_results is loaded
# Group by image_id ending with 'b' (black) or 'w' (white)

# Add a column for mask type
results['bg_color'] = results['image_id'].apply(lambda x: 'black' if str(x).endswith('b') else 'white' if str(x).endswith('w') else 'unknown')

# Filter out unknown if any
black_results = results[results['bg_color'] == 'black']
white_results = results[results['bg_color'] == 'white']

# Compute accuracy for black
if not black_results.empty:
    accuracy_black = accuracy_score(black_results['ground_truth'], black_results['prediction'])
    print(f"Accuracy for black images: {accuracy_black:.4f}")
    
    cm_black = pd.crosstab(
        black_results["ground_truth"],
        black_results["prediction"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    ).reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    
    plotter.plot_confusion_matrix(cm_black, filename="confusion_matrix_black.png")
else:
    print("No black images found.")

# Compute accuracy for white
if not white_results.empty:
    accuracy_white = accuracy_score(white_results['ground_truth'], white_results['prediction'])
    print(f"Accuracy for white images: {accuracy_white:.4f}")
    
    cm_white = pd.crosstab(
        white_results["ground_truth"],
        white_results["prediction"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    ).reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    
    plotter.plot_confusion_matrix(cm_white, filename="confusion_matrix_white.png")
else:
    print("No white images found.")

Accuracy for black images: 0.5167


Accuracy for white images: 0.5095


# How many image tokens and text tokens do we have per question, on average?

In [8]:
print("Hello world!")

Hello world!


In [9]:
# Read the CSV file
original_csv_path = "../results_llava-hf/llava-1.5-7b-hf/compute_accuracy/simple_results_all_levels.csv"
unmasked_df = pd.read_csv(original_csv_path)

# Compute the average number of image tokens and text tokens per question
avg_image_tokens = unmasked_df["num_image_tokens"].mean()
avg_text_tokens = unmasked_df["num_text_tokens"].mean()

print(f"Average number of image tokens per question: {avg_image_tokens:.2f}")
print(f"Average number of text tokens per question: {avg_text_tokens:.2f}")

Average number of image tokens per question: 576.00
Average number of text tokens per question: 52.86


# How does the model respond when we mask the image with either plain black or plain white?

In [8]:
masked_csv_path = "../results_llava-hf/llava-1.5-7b-hf/compute_accuracy/simple_results_all_levels_with_masked_images.csv"
masked_results = pd.read_csv(masked_csv_path)
print(masked_results.shape)

(3352, 9)


In [9]:


# Compute accuracy
accuracy = accuracy_score(masked_results['ground_truth'], masked_results['prediction'])
print(f"Overall Accuracy: {accuracy:.4f}")

# Compute confusion matrix
cm_all = pd.crosstab(
        masked_results["ground_truth"],
        masked_results["prediction"],
        rownames=["ground_truth"],
        colnames=["prediction"],
        dropna=False,
    ).reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)

# Plot using Plotter
plotter = Plotter(experiment_name="compute_accuracy")
plotter.plot_confusion_matrix(cm_all, filename="confusion_matrix_masked.png")

Overall Accuracy: 0.5006
